# A Real Decision Tree (Diagnosing Illness)

**Author:** [firstname] [lastname]

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

The code from the slide deck: the same patients, rules and test cases
as in the by-hand exercise, but scikit-learn builds the tree from data.
Run the cells from top to bottom.

Patient data and rules: R. Roiger, *Data Mining: A Tutorial-Based
Primer*, 2e (2021).

# Part A: The original two-symptom tree

## 1. Encode the hand-derived rules as training data

Two binary symptoms give exactly four possible patients. That is the
whole training set, no sampling needed.

In [ ]:
import pandas as pd

data = pd.DataFrame([
    {"swollen_glands": "Yes", "fever": "No",  "diagnosis": "strep throat"},
    {"swollen_glands": "Yes", "fever": "Yes", "diagnosis": "strep throat"},
    {"swollen_glands": "No",  "fever": "Yes", "diagnosis": "cold"},
    {"swollen_glands": "No",  "fever": "No",  "diagnosis": "allergy"},
])
print(data)

## 2. Build the decision tree (model-based learning)

scikit-learn needs numbers, not Yes/No strings, so the symptoms get
encoded as 1/0 first.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

X = data[["swollen_glands", "fever"]].replace({"Yes": 1, "No": 0})
y = data["diagnosis"]

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X, y)
print("Tree trained.")

## 3. Read off the production rules

Compare this directly to the three rules written by hand.

In [ ]:
from sklearn.tree import export_text

print(export_text(tree, feature_names=["swollen_glands", "fever"]))

## 4. The same tree, drawn out

Each box shows the split and `gini`, the "impurity": roughly, how mixed
the group still is (0 = perfectly pure, one diagnosis only).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(8, 5))
plot_tree(tree, feature_names=["swollen_glands", "fever"],
          class_names=sorted(y.unique()), filled=True)
plt.tight_layout()
plt.savefig("diagnosis_tree.png")
plt.show()

## 5. Test it on patients 14-17

Answer key: strep throat, allergy, cold, strep throat.

In [ ]:
patients = pd.DataFrame([
    {"patient": 14, "swollen_glands": "Yes", "fever": "No"},
    {"patient": 15, "swollen_glands": "No",  "fever": "No"},
    {"patient": 16, "swollen_glands": "No",  "fever": "Yes"},
    {"patient": 17, "swollen_glands": "Yes", "fever": "Yes"},
])
X_new = patients[["swollen_glands", "fever"]].replace({"Yes": 1, "No": 0})
patients["diagnosis (model)"] = tree.predict(X_new)
print(patients.to_string(index=False))

# Part B: Adding the duration attribute

## 6. Encode the six updated production rules as training data

Add duration (short = acute, long = chronic). Three binary attributes
give eight possible patients: again the whole space, not a sample.

In [ ]:
from itertools import product

def diagnose(glands, fever, duration):
    if glands == "Yes":
        base = "strep throat"
    elif fever == "Yes":
        base = "cold"
    else:
        base = "allergy"
    stage = "acute" if duration == "short" else "chronic"
    return f"{stage} {base}"

rows = []
for glands, fever, duration in product(["Yes", "No"], ["Yes", "No"], ["short", "long"]):
    rows.append({"swollen_glands": glands, "fever": fever, "duration": duration,
                 "diagnosis": diagnose(glands, fever, duration)})
data2 = pd.DataFrame(rows)
print(data2.to_string(index=False))

## 7. Build the tree and read off the rules

In [ ]:
X2 = data2[["swollen_glands", "fever", "duration"]].replace(
    {"Yes": 1, "No": 0, "short": 1, "long": 0})
y2 = data2["diagnosis"]

tree2 = DecisionTreeClassifier(random_state=42)
tree2.fit(X2, y2)
print(export_text(tree2, feature_names=["swollen_glands", "fever", "duration"]))

## 8. The extended tree, drawn out

Same `gini` reading as before, now over six classes instead of three.

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree2, feature_names=["swollen_glands", "fever", "duration"],
          class_names=sorted(y2.unique()), filled=True)
plt.tight_layout()
plt.savefig("diagnosis_tree_duration.png")
plt.show()